In [1]:
import pandas as pd
import numpy as np

# 加载数据（这个数据集有两个sheet）
df1 = pd.read_excel("../data/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df2 = pd.read_excel("../data/online_retail_II.xlsx", sheet_name="Year 2010-2011")
df = pd.concat([df1, df2], ignore_index=True)

print(f"数据量：{df.shape[0]:,} 行, {df.shape[1]} 列")
df.head()

数据量：1,067,371 行, 8 列


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [3]:
print("=== 缺失值统计 ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({"缺失数量": missing, "缺失比例%": missing_pct})[missing > 0])

print(f"\n=== 关键数字 ===")
print(f"总行数：{len(df):,}")
print(f"Customer ID 缺失：{df['Customer ID'].isnull().sum():,} 行 ({df['Customer ID'].isnull().mean():.1%})")
print(f"退货订单(C开头)：{df['Invoice'].astype(str).str.startswith('C').sum():,} 行")
print(f"Quantity <= 0：{(df['Quantity'] <= 0).sum():,} 行")
print(f"Price <= 0：{(df['Price'] <= 0).sum():,} 行")

=== 缺失值统计 ===
               缺失数量  缺失比例%
Description    4382   0.41
Customer ID  243007  22.77

=== 关键数字 ===
总行数：1,067,371
Customer ID 缺失：243,007 行 (22.8%)
退货订单(C开头)：19,494 行
Quantity <= 0：22,950 行
Price <= 0：6,207 行


## 数据质量发现

| 问题 | 数量 | 占比 | 处理策略 |
|------|------|------|---------|
| Customer ID 缺失 | 243,007 | 22.8% | 删除（无法做用户分层） |
| 退货订单（Invoice以C开头） | 19,494 | 1.8% | 删除（不纳入消费行为分析） |
| Quantity ≤ 0 | 22,950 | 2.1% | 删除（含退货和异常记录） |
| Price ≤ 0 | 6,207 | 0.6% | 删除（免费/异常商品） |
| Description 缺失 | 4,382 | 0.4% | 保留（不影响核心分析） |

> 注意：以上问题有重叠（比如退货订单的Quantity也是负数），实际删除行数会少于简单求和。

In [4]:
print(f"清洗前：{len(df):,} 行")

# 1. 删除 Customer ID 缺失（没有客户ID无法做用户分层）
df = df.dropna(subset=["Customer ID"])
print(f"删除 Customer ID 缺失后：{len(df):,} 行")

# 2. 删除退货订单
df = df[~df["Invoice"].astype(str).str.startswith("C")]
print(f"删除退货订单后：{len(df):,} 行")

# 3. 删除 Quantity <= 0 和 Price <= 0
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
print(f"删除异常值后：{len(df):,} 行")

# 4. 创建 Revenue 字段
df["Revenue"] = df["Quantity"] * df["Price"]

# 5. Customer ID 转整数
df["Customer ID"] = df["Customer ID"].astype(int)

print(f"\n最终保留：{len(df):,} 行")
df.head()

清洗前：1,067,371 行
删除 Customer ID 缺失后：824,364 行
删除退货订单后：805,620 行
删除异常值后：805,549 行

最终保留：805,549 行


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [5]:
print(f"=== 清洗后数据概览 ===")
print(f"订单数：{df['Invoice'].nunique():,}")
print(f"客户数：{df['Customer ID'].nunique():,}")
print(f"商品数：{df['StockCode'].nunique():,}")
print(f"国家数：{df['Country'].nunique()}")
print(f"时间范围：{df['InvoiceDate'].min().date()} ~ {df['InvoiceDate'].max().date()}")
print(f"总收入：£{df['Revenue'].sum():,.0f}")

=== 清洗后数据概览 ===
订单数：36,969
客户数：5,878
商品数：4,631
国家数：41
时间范围：2009-12-01 ~ 2011-12-09
总收入：£17,743,429


In [6]:
df.to_csv("../data/retail_cleaned.csv", index=False)
print("已保存至 data/retail_cleaned.csv")

已保存至 data/retail_cleaned.csv
